# 06 — SAE Feature Interpretation & Visualization

Interpret the Sparse Autoencoder trained in `06_training_sae.ipynb`.  
**Input**: SAE checkpoint + activation H5 file (from `06_collect_activations.ipynb`) + fine-tuned ViT checkpoint.  
**Output**: feature statistics, top-K patch visualizations, spatial heatmaps, UMAP, ablation analysis.

## 0. Kaggle Setup

In [ ]:
!rm -rf /kaggle/working/xai-vit-medical
!git clone https://github.com/youssef-nouiouar/xai-vit-medical.git /kaggle/working/xai-vit-medical
!pip install -q timm albumentations loguru h5py umap-learn
!pip install -q PyDrive2 scikit-learn

## Google Drive

In [ ]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

## Paths

In [ ]:
import os
from pathlib import Path

# ← update these
H5_PATH      = '/kaggle/input/datasets/youssefnouiouar1/deit-base-couche9-activations/deit_base_patch16_layer9_acts.h5'
SAE_CKPT     = '/kaggle/input/models/youssefnouiouar1/sae-deit-base-patch16-layer9-best/pytorch/default/1/sae_deit_base_patch16_layer9_best.pt'
VIT_CKPT     = '/kaggle/input/models/youssefnouiouar1/crt-deit/pytorch/default/1/deit_base_patch16_best.pth'
TRAINVAL_ROOT = Path('/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/NCT-CRC-HE-100K/NCT-CRC-HE-100K')
TEST_ROOT     = Path('/kaggle/input/datasets/youssefnouiouar1/colorectal-cancer-histology-nct-crc-he-100k-and-7k/CRC-VAL-HE-7K/CRC-VAL-HE-7K')
PROJECT_ROOT  = '/kaggle/working/xai-vit-medical'

SAVE_DIR = '/kaggle/working/interpretation'
os.makedirs(SAVE_DIR, exist_ok=True)

## 1. Imports & Config

In [ ]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import h5py
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image
from tqdm.notebook import tqdm
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE   # replaces umap — no Numba JIT, runs instantly
import timm

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.crc_dataset import CRCHistologyDataset, CRCSplits, DEFAULT_CRC_CLASSES
from src.utils.seed import set_seed

SEED        = 42
set_seed(SEED)
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── SAE / model params ────────────────────────────────────────────────────────
D_INPUT      = 768
D_SAE        = 768 * 16    # 12 288
K            = 32
LAYER_IDX    = 9
NUM_CLASSES  = 9
CLASS_NAMES  = list(DEFAULT_CRC_CLASSES)
PATCH_GRID   = 14           # 14×14 grid (DeiT/ViT-Base with 16px patches)
PATCH_SIZE   = 16           # pixels per patch
IMAGE_SIZE   = 224

# visualization subset
SAMPLES_PER_CLASS = 100      # images per class for visualization
TOP_FEATS_PER_CLASS = 3     # how many top features to inspect per class

print(f'Device : {DEVICE}')

## 2. Load SAE

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d_input, d_sae, k):
        super().__init__()
        self.d_input = d_input
        self.d_sae   = d_sae
        self.k       = k
        self.W_enc = nn.Parameter(torch.empty(d_input, d_sae))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.empty(d_sae,  d_input))
        self.b_dec = nn.Parameter(torch.zeros(d_input))

    def encode(self, x):
        pre  = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre)
        acts.scatter_(-1, idx, vals.clamp(min=0))
        return acts

    def decode(self, acts):
        return acts @ self.W_dec + self.b_dec

    def forward(self, x):
        acts  = self.encode(x)
        return self.decode(acts), acts


ckpt_sae = torch.load(SAE_CKPT, map_location='cpu')
cfg      = ckpt_sae['config']

sae = TopKSAE(cfg['d_input'], cfg['d_sae'], cfg['k'])
sae.W_enc.data = ckpt_sae['W_enc']
sae.b_enc.data = ckpt_sae['b_enc']
sae.W_dec.data = ckpt_sae['W_dec']
sae.b_dec.data = ckpt_sae['b_dec']
sae = sae.to(DEVICE).eval()

MEAN = ckpt_sae['mean'].to(DEVICE)   # (768,) normalization mean
STD  = ckpt_sae['std'].to(DEVICE)    # (768,) normalization std

print(f'SAE loaded  : d_input={cfg["d_input"]}  d_sae={cfg["d_sae"]}  k={cfg["k"]}')
print(f'Best epoch  : step {ckpt_sae["step"]}')

## 3. Streaming Metrics over H5

Single pass over all activations to compute **reconstruction quality** (R², cosine similarity) and **feature activity** (activation frequency, class specificity).  
Nothing is fully loaded into RAM — processed in batches of 256 images.

In [ ]:
BATCH_IMG = 256

feat_fires       = torch.zeros(D_SAE)
class_fires      = torch.zeros(NUM_CLASSES, D_SAE)
class_count_tens = torch.zeros(NUM_CLASSES)
ss_res = ss_tot = cos_sum = n_tokens = 0.0

with h5py.File(H5_PATH, 'r') as f:
    N_img, P, D = f['activations'].shape
    labels_h5   = f['labels'][:]

    for i in tqdm(range(0, N_img, BATCH_IMG), desc='Metrics pass'):
        acts_np  = f['activations'][i:i+BATCH_IMG].astype(np.float32)  # (B, 196, 768)
        labels_b = labels_h5[i:i+BATCH_IMG]
        B        = len(acts_np)

        tokens = torch.from_numpy(acts_np.reshape(B*P, D)).to(DEVICE)
        x      = (tokens - MEAN) / STD

        with torch.no_grad():
            codes = sae.encode(x)      # (B*P, D_SAE)
            x_hat = sae.decode(codes)  # (B*P, 768)

        fires = (codes > 0).float()    # (B*P, D_SAE) on GPU
        feat_fires += fires.sum(dim=0).cpu()

        # class-level counts via one-hot scatter
        labels_rep   = torch.from_numpy(np.repeat(labels_b, P)).long()
        onehot       = F.one_hot(labels_rep, NUM_CLASSES).float()   # (B*P, 9)
        class_fires += (onehot.T @ fires.cpu())                     # (9, D_SAE)
        class_count_tens += onehot.sum(dim=0)

        ss_res  += ((x - x_hat)**2).sum().item()
        ss_tot  += ((x - x.mean(0, keepdim=True))**2).sum().item()
        cos_sum += F.cosine_similarity(x, x_hat).sum().item()
        n_tokens += B * P

        del tokens, x, codes, x_hat, fires

feat_freq  = (feat_fires / n_tokens).numpy()                              # (D_SAE,)
class_freq = (class_fires / class_count_tens.unsqueeze(1)).numpy()        # (9, D_SAE)
r2         = float(1 - ss_res / (ss_tot + 1e-8))
cos_sim    = float(cos_sum / n_tokens)
dead_pct   = float((feat_freq < 1e-4).mean() * 100)

print(f'R²              : {r2:.4f}   (target > 0.85)')
print(f'Cosine sim      : {cos_sim:.4f}   (target > 0.90)')
print(f'Dead features   : {dead_pct:.1f}%   (healthy: 10-30%)')
print(f'Tokens processed: {int(n_tokens):,}')

## 4. Class Selectivity & Top Features per Class

In [ ]:
# class_freq: (9, D_SAE) — activation freq of each feature per class
best_class = class_freq.argmax(axis=0)      # (D_SAE,) — which class activates each feature most
max_freq   = class_freq.max(axis=0)         # (D_SAE,)

# mean activation freq across the OTHER classes
mean_other = (class_freq.sum(axis=0) - max_freq) / (NUM_CLASSES - 1)

selectivity = (max_freq - mean_other) / (max_freq + mean_other + 1e-8)  # (D_SAE,)

# Top features per class
# BUG FIX: rank within class-c features only.
# The old approach (selectivity * class_mask) gives score=0 to non-class-c features.
# When all class-c features have negative selectivity they rank BELOW the zeros,
# so argsort picked random non-class-c features instead.
top_features_per_class = {}
features_to_viz = set()

print(f'{"Class":6s}  {"Top feat":>10s}  {"selectivity":>12s}  {"freq":>8s}')
print('-' * 44)
for c, cname in enumerate(CLASS_NAMES):
    # Only consider features whose best class is c
    class_feat_idx = np.where(best_class == c)[0]   # (n_c,) feature indices

    if len(class_feat_idx) > 0:
        order   = np.argsort(selectivity[class_feat_idx])[::-1][:TOP_FEATS_PER_CLASS]
        top_idx = class_feat_idx[order]
    else:
        # Fallback: features that fire most often for this class (no argmax winner)
        top_idx = np.argsort(class_freq[c])[::-1][:TOP_FEATS_PER_CLASS]

    top_features_per_class[cname] = top_idx
    features_to_viz.update(top_idx.tolist())
    print(f'{cname:6s}  feat #{top_idx[0]:>6d}   sel={selectivity[top_idx[0]]:.3f}   freq={max_freq[top_idx[0]]:.4f}')

features_to_viz = sorted(features_to_viz)
print(f'\nTotal features to visualize: {len(features_to_viz)}')

## 5. Feature Statistics Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Activation frequency
freq_pct = feat_freq * 100
axes[0].hist(freq_pct[freq_pct > 0], bins=80, color='steelblue', edgecolor='none')
axes[0].axvline(0.01, color='red',    ls='--', label='0.01% near-dead')
axes[0].axvline(5.0,  color='orange', ls='--', label='5% universal')
axes[0].set(xlabel='Activation frequency (%)', ylabel='# Features',
            title='Feature Activation Frequency', xscale='log')
axes[0].legend(fontsize=8)

# Class selectivity
axes[1].hist(selectivity[selectivity > 0], bins=60, color='seagreen', edgecolor='none')
axes[1].axvline(0.5, color='red', ls='--', label='sel > 0.5')
axes[1].set(xlabel='Selectivity index', ylabel='# Features', title='Class Selectivity')
axes[1].legend(fontsize=8)

# Top-1 feature per class — show selectivity and annotate with feature index
top1_sel  = [selectivity[top_features_per_class[c][0]] for c in CLASS_NAMES]
top1_feat = [int(top_features_per_class[c][0])         for c in CLASS_NAMES]
bars = axes[2].barh(CLASS_NAMES, top1_sel, color='tomato')
for bar, feat_i, sel_val in zip(bars, top1_feat, top1_sel):
    x_label = sel_val + 0.01 if sel_val >= 0 else 0.01
    axes[2].text(x_label, bar.get_y() + bar.get_height() / 2,
                 f'feat #{feat_i}', va='center', fontsize=7)
axes[2].set(xlabel='Selectivity (top-1 feature)', title='Top Feature per Class')
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'feature_stats.png'), dpi=150)
plt.show()

## 6. Downstream Task Recovery

Train a logistic regression on SAE-reconstructed activations vs. original activations.  
Measures how much diagnostic information the SAE preserves.

In [ ]:
X_orig  = []
X_recon = []
y_probe = []

with h5py.File(H5_PATH, 'r') as f:
    N_img, P, D = f['activations'].shape
    labels_h5   = f['labels'][:]

    for i in tqdm(range(0, N_img, BATCH_IMG), desc='Probe features'):
        acts_np  = f['activations'][i:i+BATCH_IMG].astype(np.float32)
        B        = len(acts_np)

        tokens = torch.from_numpy(acts_np.reshape(B*P, D)).to(DEVICE)
        x      = (tokens - MEAN) / STD

        with torch.no_grad():
            codes = sae.encode(x)
            x_hat = sae.decode(codes)

        # mean-pool over patches → image-level feature
        X_orig.append(x.reshape(B, P, D).mean(1).cpu().numpy())       # (B, 768)
        X_recon.append(x_hat.reshape(B, P, D).mean(1).cpu().numpy())   # (B, 768)
        y_probe.extend(labels_h5[i:i+BATCH_IMG].tolist())

        del tokens, x, codes, x_hat

X_orig  = np.vstack(X_orig)
X_recon = np.vstack(X_recon)
y_probe = np.array(y_probe)

# train on 80%, test on 20%
from sklearn.model_selection import train_test_split
idx_tr, idx_te = train_test_split(np.arange(len(y_probe)), test_size=0.2,
                                   random_state=SEED, stratify=y_probe)

clf_orig  = LogisticRegression(max_iter=500, C=1.0, n_jobs=-1)
clf_recon = LogisticRegression(max_iter=500, C=1.0, n_jobs=-1)
clf_orig.fit(X_orig[idx_tr],  y_probe[idx_tr])
clf_recon.fit(X_recon[idx_tr], y_probe[idx_tr])

acc_orig  = accuracy_score(y_probe[idx_te], clf_orig.predict(X_orig[idx_te]))
acc_recon = accuracy_score(y_probe[idx_te], clf_recon.predict(X_recon[idx_te]))

print(f'Linear probe — original activations : {acc_orig:.4f}')
print(f'Linear probe — SAE reconstructions  : {acc_recon:.4f}')
print(f'Information gap                      : {(acc_orig - acc_recon)*100:.1f}pp  (target < 3%)')

## 7. Visualization — Load ViT + Sample Images

We sample `SAMPLES_PER_CLASS` images per class from the val split, run them through the fine-tuned ViT + SAE,  
and store activations for the features identified in Section 4.

In [ ]:
# Load fine-tuned ViT
model = timm.create_model('deit_base_patch16_224', pretrained=False, num_classes=NUM_CLASSES)
ckpt_vit = torch.load(VIT_CKPT, map_location='cpu')
model.load_state_dict(ckpt_vit['model_state_dict'])
model = model.to(DEVICE).eval()
print('ViT loaded.')

# Hook on layer LAYER_IDX to capture patch-token activations
act_buffer = []
def hook_fn(module, inp, out):
    act_buffer.append(out[:, 1:, :].cpu())   # skip CLS token → (B, 196, 768)

hook = model.blocks[LAYER_IDX].register_forward_hook(hook_fn)

# Val dataset — same params as collection notebook
crc_splits = CRCSplits(
    trainval_root=TRAINVAL_ROOT,
    test_root=TEST_ROOT,
    classes=tuple(CLASS_NAMES),
    val_ratio=0.5,
    random_state=SEED,
)
val_ds = CRCHistologyDataset(split='val', splits=crc_splits,
                              image_size=IMAGE_SIZE, return_id=True)

# Sample SAMPLES_PER_CLASS per class
class_idx_map = defaultdict(list)
for i, lbl in enumerate(val_ds.labels):
    class_idx_map[int(lbl)].append(i)

rng_vis = np.random.default_rng(SEED + 1)
vis_indices = []
for c in range(NUM_CLASSES):
    chosen = rng_vis.choice(class_idx_map[c], SAMPLES_PER_CLASS, replace=False)
    vis_indices.extend(chosen.tolist())

vis_loader = DataLoader(Subset(val_ds, vis_indices),
                        batch_size=32, shuffle=False, num_workers=2)
print(f'Vis set: {len(vis_indices)} images  ({SAMPLES_PER_CLASS} per class)')

In [ ]:
feat_idx_t = torch.tensor(features_to_viz, dtype=torch.long)  # indices to store
n_viz      = len(features_to_viz)

vis_images_np   = []   # (N_vis, 224, 224, 3) uint8
vis_labels_list = []
vis_feat_acts   = []   # (N_vis, 196, n_viz)  — only top features
vis_mean_codes  = []   # (N_vis, D_SAE)       — for UMAP
vis_recon_mean  = []   # (N_vis, 768)         — for ablation

_imagenet_mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
_imagenet_std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

with torch.no_grad():
    for imgs, lbls, _ in tqdm(vis_loader, desc='Forward pass'):
        imgs = imgs.to(DEVICE)
        B    = len(imgs)

        act_buffer.clear()
        model(imgs)
        layer_acts = act_buffer[0].to(DEVICE)   # (B, 196, 768)

        x     = (layer_acts.reshape(B*196, 768) - MEAN) / STD
        codes = sae.encode(x)                    # (B*196, D_SAE) on GPU
        x_hat = sae.decode(codes)                # (B*196, 768)

        codes3d = codes.view(B, 196, D_SAE)

        vis_feat_acts.append(codes3d[:, :, feat_idx_t].cpu().numpy())  # (B, 196, n_viz)
        vis_mean_codes.append(codes3d.mean(dim=1).cpu().numpy())       # (B, D_SAE)
        vis_recon_mean.append(x_hat.view(B, 196, 768).mean(1).cpu().numpy())  # (B, 768)

        imgs_disp = (imgs.cpu() * _imagenet_std + _imagenet_mean).clamp(0, 1)
        vis_images_np.extend((imgs_disp.permute(0,2,3,1).numpy()*255).astype(np.uint8))
        vis_labels_list.extend(lbls.tolist())

        del codes, x_hat, codes3d, x

hook.remove()

vis_feat_acts  = np.concatenate(vis_feat_acts,  axis=0)   # (N_vis, 196, n_viz)
vis_mean_codes = np.concatenate(vis_mean_codes, axis=0)   # (N_vis, D_SAE)
vis_recon_mean = np.concatenate(vis_recon_mean, axis=0)   # (N_vis, 768)
vis_labels_arr = np.array(vis_labels_list)
print(f'Collected: {vis_feat_acts.shape}  |  mean_codes: {vis_mean_codes.shape}')

## 8. Top-K Activating Patches

For each of the top class-specific features, show the 9 patches that activate it most strongly.  
If patches for one feature are visually coherent (all glandular lumens, all lymphocytes…) → **monosemantic**.

In [ ]:
def show_top_patches(feature_idx, k=9, title=None):
    """Display top-k patches that maximally activate a given feature."""
    if feature_idx not in features_to_viz:
        print(f'Feature {feature_idx} not in visualization set. Add it to features_to_viz.')
        return

    col      = features_to_viz.index(feature_idx)
    flat     = vis_feat_acts[:, :, col].reshape(-1)   # (N_vis * 196,)
    top_flat = np.argsort(flat)[::-1][:k]

    nrows = (k + 2) // 3
    fig, axes = plt.subplots(nrows, 3, figsize=(7, nrows * 2.5))
    axes = axes.flatten()

    for rank, flat_i in enumerate(top_flat):
        img_i   = flat_i // 196
        patch_i = flat_i %  196
        row_p   = patch_i // PATCH_GRID
        col_p   = patch_i %  PATCH_GRID
        y1, x1  = row_p * PATCH_SIZE, col_p * PATCH_SIZE
        patch   = vis_images_np[img_i][y1:y1+PATCH_SIZE, x1:x1+PATCH_SIZE]

        axes[rank].imshow(patch)
        axes[rank].set_title(f'act={flat[flat_i]:.4f}\n{CLASS_NAMES[vis_labels_arr[img_i]]}',
                              fontsize=7)
        axes[rank].axis('off')

    for ax in axes[k:]:
        ax.axis('off')

    suptitle = title or f'Feature #{feature_idx} — best class: {CLASS_NAMES[best_class[feature_idx]]}  sel={selectivity[feature_idx]:.3f}'
    plt.suptitle(suptitle, fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'topk_feat{feature_idx}.png'), dpi=150)
    plt.show()


# Show top-1 feature for each class
for cname in CLASS_NAMES:
    feat = top_features_per_class[cname][0]
    show_top_patches(feat, k=9, title=f'[{cname}] Feature #{feat}')

## 9. Spatial Heatmaps

For a given image and feature, reshape the 196 patch activations into a 14×14 grid  
and overlay as a heatmap on the original H&E image.

In [ ]:
def spatial_heatmap(img_np, acts_196, ax, alpha=0.45):
    """Overlay feature activation heatmap on image."""
    hm     = acts_196.reshape(PATCH_GRID, PATCH_GRID)
    hm_up  = np.kron(hm, np.ones((PATCH_SIZE, PATCH_SIZE)))    # 224×224
    hm_norm = (hm_up - hm_up.min()) / (hm_up.max() - hm_up.min() + 1e-8)
    ax.imshow(img_np)
    ax.imshow(cm.hot(hm_norm), alpha=alpha)
    ax.axis('off')


# For each class: show 3 sample images with the top-1 class feature heatmap
N_EXAMPLES = 3

for cname in CLASS_NAMES:
    feat    = top_features_per_class[cname][0]
    col     = features_to_viz.index(feat)
    c_idx   = CLASS_NAMES.index(cname)
    c_imgs  = np.where(vis_labels_arr == c_idx)[0][:N_EXAMPLES]

    fig, axes = plt.subplots(1, N_EXAMPLES, figsize=(N_EXAMPLES * 4, 4))
    for ax, img_i in zip(axes, c_imgs):
        spatial_heatmap(vis_images_np[img_i],
                        vis_feat_acts[img_i, :, col],
                        ax)
    plt.suptitle(f'[{cname}] Feature #{feat} — sel={selectivity[feat]:.3f}', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'heatmap_{cname}_feat{feat}.png'), dpi=150)
    plt.show()

## 10. t-SNE of SAE Feature Space

Each image is represented by its mean SAE code (12 288-dim).  
PCA → 50 dims first (noise reduction), then t-SNE → 2D.  
Color by tissue class: if TUM clusters separately from NORM, the SAE captures diagnostically relevant variation.

### UMAP of Feature Space — Detailed Explanation

**What is "each image's SAE code"?**

After training, the SAE encodes every patch token as a sparse vector of 12,288 dimensions, where exactly 32 values are nonzero (because k=32). That sparse vector is called the **SAE code** for that token. For example:

```
[0, 0, 0, 4.2, 0, 0, 1.7, 0, ..., 3.1, 0, ...]
 ^              ^          ^          ^
 dead         feat#3    feat#6     feat#9001
              fires      fires      fires
```

That is a 12,288-dimensional vector with only 32 nonzero entries.

**Why UMAP?**

You cannot visualize 12,288 dimensions directly. UMAP compresses them to 2D while preserving neighborhood structure: patch tokens that are "close" in 12,288D stay close in 2D. Each point in the plot = one patch token (or one image if you average its patches), colored by tissue class.

```
      UMAP dim 2
          │      ● ● ●  ← TUM cluster
          │    ● ● ●
          │
          │  ■ ■ ■      ← NORM cluster
          │■ ■ ■
          │
          │    ▲ ▲      ← STR cluster
          └──────────── UMAP dim 1
```

**How to read the result:**

| What you see | What it means |
|---|---|
| Tight, separated clusters by class | SAE features capture diagnostically relevant differences — the 32 active features for TUM patches are consistently different from those for NORM patches |
| All classes mixed together | SAE features are generic (staining, texture) — they don't encode class-relevant biology; adjust k or d_sae |
| A class split into two sub-clusters | SAE may have discovered two biological subtypes within one class |

**Why this matters for the thesis:** The original ViT activation space (768D) likely already clusters by class — it was trained to classify. The real question is whether the SAE's sparse code (only 32 selected features) still preserves that structure. If yes → sparse features are sufficient to distinguish tissue types → they are diagnostically meaningful. If no → the SAE is capturing something orthogonal (staining artifacts, scanner variation) → the expansion factor or k needs adjustment. Run this visualization before any detailed per-feature analysis as a global sanity check.

> **Why t-SNE instead of UMAP?** UMAP uses Numba JIT compilation which hangs indefinitely on Kaggle GPU kernels (low CPU/GPU usage, no output). t-SNE is pure C via sklearn — no compilation, runs in seconds for < 500 samples.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import os
import umap

## save the varaible for needed for UMAP
# np.save("vis_mean_codes.npy",vis_mean_codes)
# np.save("vis_labels_arr.npy",vis_labels_arr)
## load the variable 
vis_mean_codes=np.load("vis_mean_codes.npy")
vis_labels_arr=np.load("vis_labels_arr.npy")
CLASS_NAMES=['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
SEED=42
NUM_CLASSES=9

X = vis_mean_codes   # (N_vis, D_SAE)
N_vis = len(X)

# n_components must be < n_samples
n_pca      = min(50, N_vis - 1)
# t-SNE perplexity must be < n_samples; typical range 5–50
perplexity = min(30, max(5, N_vis // 4))

print(f'N_vis={N_vis}  →  n_pca={n_pca}  perplexity={perplexity}')

# svd_solver='randomized': computes only the first n_pca singular vectors.
# Default 'full' on (N, 12288) allocates a full (12288, 12288) Vt matrix
# (~600 MB) even though we only need 50 columns → kernel freeze on Kaggle.
X_pca = PCA(n_components=n_pca, svd_solver='full',
            random_state=SEED).fit_transform(X)

X_2d = TSNE(n_components=2, random_state=SEED,
             perplexity=perplexity, max_iter=1000).fit_transform(X_pca)

colors = plt.get_cmap('tab10')(np.linspace(0, 1, NUM_CLASSES))

fig, ax = plt.subplots(figsize=(10, 8))
for c, cname in enumerate(CLASS_NAMES):
    mask = vis_labels_arr == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               label=cname, color=colors[c], alpha=0.7, s=40, edgecolors='none')
ax.legend(markerscale=2, fontsize=9)
ax.set(title='t-SNE — SAE Feature Space (mean SAE code per image)',
       xlabel='t-SNE-1', ylabel='t-SNE-2')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'tsne_sae_features.png'), dpi=150)
plt.show()

## 11. Ablation Test — Activation Patching

For the top feature of a class:
1. Run the image through the model, save layer-9 activations
2. Encode activations with the SAE → zero the feature → decode back
3. Inject modified activations, re-run layers 10-11 + head
4. Compare prediction before / after

If `TUM → NORM` after ablation: feature is **causally important**.
If prediction unchanged: feature is **correlational only** (low freq or redundancy).

In [ ]:
CLASS_TO_ABLATE = 'TUM'   # <- change to inspect other classes
N_FEATS_ABLATE  = 40

# ── 1. Setup ──────────────────────────────────────────────────────────────
feats_ablate = [int(f) for f in top_features_per_class[CLASS_TO_ABLATE][:N_FEATS_ABLATE]]
c_ablate     = CLASS_NAMES.index(CLASS_TO_ABLATE)
N_SPECIAL    = 2 if 'distilled' in MODEL_LABEL.lower() else 1

print(f'Class  : {CLASS_TO_ABLATE}  (index {c_ablate})')
print(f'N feats: {N_FEATS_ABLATE}  |  N_SPECIAL: {N_SPECIAL}')
print(f'Features: {feats_ablate}')

# ── 2. Forward pass: collect y9 + baseline predictions ───────────────────
model.eval()
y9_list    = []
base_preds = []

for imgs, labels, _ in tqdm(vis_loader, desc='Collect y9'):
    mask = (labels == c_ablate)
    if mask.sum() == 0:
        continue
    imgs = imgs[mask].to(DEVICE)

    y9_store = {}
    def _save(m, inp, out):
        y9_store['y9'] = out.detach().clone()
    h = model.blocks[LAYER_IDX].register_forward_hook(_save)
    with torch.no_grad():
        logits = model(imgs)
    h.remove()

    y9_list.append(y9_store['y9'].cpu())
    base_preds.extend(logits.argmax(1).cpu().tolist())

y9_all     = torch.cat(y9_list, dim=0)
base_preds = np.array(base_preds)
N_cls      = len(y9_all)
N_P        = y9_all.shape[1] - N_SPECIAL
D          = y9_all.shape[2]

# ── Pre-check: model accuracy on this class ───────────────────────────────
correct_mask = (base_preds == c_ablate)
n_correct    = correct_mask.sum()
accuracy     = n_correct / N_cls * 100
pred_dist    = dict(Counter(CLASS_NAMES[p] for p in base_preds))

print(f'\nModel accuracy on {CLASS_TO_ABLATE}: {n_correct}/{N_cls} ({accuracy:.1f}%)')
print(f'Prediction distribution: {pred_dist}')

if n_correct == 0:
    print(f'\nERROR: model does not classify ANY {CLASS_TO_ABLATE} image correctly.')
    print(f'Ablation is meaningless — there is nothing to flip away from {CLASS_TO_ABLATE}.')
    print(f'Check CLASS_NAMES ordering, model checkpoint, or try a different class.')
    raise RuntimeError(f'0% accuracy on {CLASS_TO_ABLATE} — cannot run ablation')

if accuracy < 50:
    print(f'WARNING: low accuracy ({accuracy:.1f}%) — ablation results may be unreliable.')

# ── Filter to correctly classified images only ────────────────────────────
y9_correct     = y9_all[correct_mask]
N_correct      = len(y9_correct)
print(f'Running ablation on {N_correct} correctly classified {CLASS_TO_ABLATE} images.')

# ── 3. Zero top-N features, inject, get new predictions ──────────────────
BATCH = 32
preds_after = []

for i in tqdm(range(0, N_correct, BATCH), desc='Ablation'):
    y9 = y9_correct[i:i+BATCH].to(DEVICE)
    B  = len(y9)

    y9_patch = y9[:, N_SPECIAL:, :]
    with torch.no_grad():
        codes = sae.encode((y9_patch.reshape(B * N_P, D) - MEAN) / STD)
    codes[:, feats_ablate] = 0.0
    with torch.no_grad():
        x_hat = sae.decode(codes) * STD + MEAN

    y9_ablated = y9.clone()
    y9_ablated[:, N_SPECIAL:, :] = x_hat.reshape(B, N_P, D)

    def _inject(m, inp, out, _y=y9_ablated):
        return _y
    h = model.blocks[LAYER_IDX].register_forward_hook(_inject)
    with torch.no_grad():
        dummy  = torch.zeros(B, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        logits = model(dummy)
    h.remove()
    preds_after.extend(logits.argmax(1).cpu().tolist())

# ── 4. Results ────────────────────────────────────────────────────────────
from collections import Counter
preds_after = np.array(preds_after)

print(f'\nSimultaneous ablation of {N_FEATS_ABLATE} features on {N_correct} {CLASS_TO_ABLATE} images:')
print(f'  Before : {dict(Counter(CLASS_NAMES[p] for p in [c_ablate]*N_correct))}')
print(f'  After  : {dict(Counter(CLASS_NAMES[p] for p in preds_after))}')

flipped  = preds_after != c_ablate
flip_pct = flipped.mean() * 100
print(f'  Flipped away from {CLASS_TO_ABLATE}: {flipped.sum()}/{N_correct} ({flip_pct:.1f}%)')

if flipped.sum() > 0:
    flip_to = dict(Counter(CLASS_NAMES[p] for p in preds_after[flipped]))
    print(f'  Flipped to: {flip_to}')
    print(f'  -> Top-{N_FEATS_ABLATE} features jointly encode {CLASS_TO_ABLATE}-specific information.')
else:
    print(f'  -> Top-{N_FEATS_ABLATE} features are NOT causally sufficient.')
    print(f'     Model uses many redundant features — try zeroing all features or a different layer.')

In [ ]:
# ── Sanity checks — verify the hook injection is working ─────────────────
# Run these BEFORE trusting the 0% flip result.
#
# Check A: inject ORIGINAL y9  →  predictions must be identical to baseline
# Check B: inject all-ZERO y9  →  predictions must change dramatically
# Check C: measure L2 distance between original and ablated patches
#          if L2 ≈ 0, the 40 features don't carry much energy

BATCH = 32
y9_batch = y9_all[:BATCH].to(DEVICE)   # use first batch for all checks
B_s      = len(y9_batch)

def run_with_injected_y9(y9_inject):
    """Run model blocks LAYER_IDX+1..end + head on an injected y9."""
    def _hook(m, inp, out, _y=y9_inject):
        return _y
    h = model.blocks[LAYER_IDX].register_forward_hook(_hook)
    with torch.no_grad():
        dummy  = torch.zeros(B_s, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        logits = model(dummy)
    h.remove()
    return logits.argmax(1).cpu().numpy()

# ── Baseline (no hook) ────────────────────────────────────────────────────
with torch.no_grad():
    dummy      = torch.zeros(B_s, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    logits_raw = model(dummy)
preds_dummy = logits_raw.argmax(1).cpu().numpy()

# ── Check A: inject original y9 — must match base_preds ──────────────────
preds_A = run_with_injected_y9(y9_batch)
match_A = (preds_A == base_preds[:B_s]).mean() * 100
print(f'[Check A] Inject original y9   →  match baseline: {match_A:.1f}%  (expect 100%)')
if match_A < 100:
    print('  WARNING: hook injection does NOT reproduce baseline — code bug!')
else:
    print('  OK: hook works correctly.')

# ── Check B: inject all-zero y9 — predictions must collapse ─────────────
y9_zeros  = torch.zeros_like(y9_batch)
preds_B   = run_with_injected_y9(y9_zeros)
match_B   = (preds_B == base_preds[:B_s]).mean() * 100
print(f'[Check B] Inject zeros as y9   →  match baseline: {match_B:.1f}%  (expect ~0-11%)')
if match_B > 50:
    print('  WARNING: predictions barely changed with zero y9 — hook may not be on the right layer!')
else:
    print('  OK: injecting zeros causes prediction collapse — hook propagates correctly.')

# ── Check C: L2 distance between original and ablated patch activations ───
y9_patch_orig = y9_batch[:, N_SPECIAL:, :].reshape(B_s * N_P, D)
with torch.no_grad():
    codes_orig = sae.encode((y9_patch_orig - MEAN) / STD)
codes_abl = codes_orig.clone()
codes_abl[:, feats_ablate] = 0.0
with torch.no_grad():
    x_hat_abl = sae.decode(codes_abl) * STD + MEAN
    x_hat_orig = sae.decode(codes_orig) * STD + MEAN

l2_ablated_vs_orig = (x_hat_abl - y9_patch_orig).norm(dim=-1).mean().item()
l2_sae_recon_orig  = (x_hat_orig - y9_patch_orig).norm(dim=-1).mean().item()
l2_feature_delta   = (x_hat_abl - x_hat_orig).norm(dim=-1).mean().item()

energy_removed_pct = (codes_orig[:, feats_ablate].abs().sum() /
                      codes_orig.abs().sum().clamp(min=1e-8) * 100).item()

print(f'[Check C] Patch activation distances (L2 per token, mean over {B_s} images):')
print(f'  SAE recon error (original):   {l2_sae_recon_orig:.4f}  (baseline SAE quality)')
print(f'  Ablation delta (abl - orig):  {l2_feature_delta:.4f}  (how much the patches changed)')
print(f'  Ablated vs ground-truth:      {l2_ablated_vs_orig:.4f}')
print(f'  Energy in zeroed features:    {energy_removed_pct:.2f}% of total SAE activation')
print()
if l2_feature_delta < 0.01:
    print('  -> The 40 features carry very little energy — ablation has near-zero effect on activations.')
    print('     This EXPLAINS the 0% flip rate: the input to blocks 10+ barely changed.')
elif match_A == 100 and match_B < 50:
    print('  -> Hook works AND ablation changes activations significantly.')
    print('     0% flip rate is a REAL RESULT: model is robust to this ablation.')
    print('     Possible reasons: redundancy across 6144 features, or layer 9 is too early.')

## 12. Upload Results to Google Drive

In [ ]:
import glob

for fpath in glob.glob(os.path.join(SAVE_DIR, '*.png')):
    gd = drive.CreateFile({'title': os.path.basename(fpath),
                          "parents": [{"id": '1HvIGjyZ-f7wK4iGKs2qr_Fwv60xaGBtw'}]
                          })
    gd.SetContentFile(fpath)
    gd.Upload()
    print(f'Uploaded: {os.path.basename(fpath)}')

print('Done.')

In [ ]:
# ── Decision Tree on SAE top features ───────────────────────────────────
# Feature per image: max activation of each top feature across 196 patches.
#   value = 0   → feature never fired on this image
#   value > 0   → feature fired; value encodes how strongly
# The tree learns: 'if feature X fires above threshold T → class C'

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

TOP_K_PER_CLASS = 10   # top features per class to include
MAX_DEPTH       = 5    # tree depth (lower = more readable)

# safe defaults if ablation cell was not run
if 'N_SPECIAL' not in dir(): N_SPECIAL = 2 if 'distilled' in MODEL_LABEL.lower() else 1
if 'N_P'       not in dir(): N_P = PATCH_GRID ** 2
if 'D'         not in dir(): D = 768

# ── 1. Select features: union of top-K across all classes ────────────────
selected_feats = []
for name in CLASS_NAMES:
    selected_feats.extend([int(f) for f in top_features_per_class[name][:TOP_K_PER_CLASS]])
selected_feats = list(dict.fromkeys(selected_feats))   # unique, keep order
F = len(selected_feats)
print(f'Top-{TOP_K_PER_CLASS} per class -> {F} unique features selected')

# ── 2. Build feature matrix (N_images, F) ────────────────────────────────
model.eval()
X_rows     = []
all_labels = []
deit_preds = []

for imgs, labels, _ in tqdm(vis_loader, desc='Encoding'):
    imgs = imgs.to(DEVICE)
    B    = len(imgs)

    y9_store = {}
    def _save(m, inp, out): y9_store['y9'] = out.detach().clone()
    h = model.blocks[LAYER_IDX].register_forward_hook(_save)
    with torch.no_grad(): logits = model(imgs)
    h.remove()

    deit_preds.extend(logits.argmax(1).cpu().tolist())
    all_labels.extend(labels.tolist())

    patches = y9_store['y9'][:, N_SPECIAL:, :].reshape(B * N_P, D)
    with torch.no_grad():
        codes_all = sae.encode((patches - MEAN) / STD)   # (B*N_P, D_SAE)

    # max over patches: 'how much does this feature fire on this image?'
    codes_img = codes_all.reshape(B, N_P, -1).max(dim=1).values  # (B, D_SAE)
    X_rows.append(codes_img[:, selected_feats].cpu().float().numpy())

X      = np.vstack(X_rows)    # (N_images, F)
y_true = np.array(all_labels)
y_deit = np.array(deit_preds)
print(f'Feature matrix: {X.shape}  |  DeiT accuracy: {(y_deit==y_true).mean()*100:.1f}%')

# ── 3. Train Decision Tree ────────────────────────────────────────────────
clf = DecisionTreeClassifier(
    max_depth=MAX_DEPTH,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
)
clf.fit(X, y_true)
y_tree = clf.predict(X)
print(f'Tree accuracy : {(y_tree==y_true).mean()*100:.1f}%  ',
      f'(depth={clf.get_depth()}, leaves={clf.get_n_leaves()})')

# ── 4. Per-class: DeiT vs Tree ────────────────────────────────────────────
print(f"\n{'Class':<6} {'N':>4} {'DeiT%':>7} {'Tree%':>7} {'Agree%':>8}  Interpretation")
print('-' * 68)
for c, name in enumerate(CLASS_NAMES):
    mask  = y_true == c
    if not mask.any(): continue
    d_acc = (y_deit[mask] == c).mean() * 100
    t_acc = (y_tree[mask] == c).mean() * 100
    agr   = (y_deit[mask] == y_tree[mask]).mean() * 100
    if   d_acc >= 80 and t_acc >= 80: note = 'consistent  (top features explain DeiT)'
    elif d_acc >= 80 and t_acc <  50: note = 'DeiT uses more than these top features'
    elif d_acc <  50 and t_acc >= 80: note = 'SAE features better than DeiT here'
    else:                             note = ''
    print(f'{name:<6} {mask.sum():>4} {d_acc:>6.1f}% {t_acc:>6.1f}% {agr:>7.1f}%  {note}')

# ── 5. Feature names: show freq_img and selectivity in each node ─────────
feat_names = []
for f in selected_feats:
    freq = class_freq_img[:, f].max() * 100          # max freq across classes
    sel  = selectivity[f]
    feat_names.append(f'f{f} freq={freq:.0f}% sel={sel:.2f}')

# ── 6. Plot the tree ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    clf,
    feature_names=feat_names,
    class_names=CLASS_NAMES,
    filled=True,
    rounded=True,
    fontsize=7,
    ax=ax,
    impurity=False,
    proportion=True,
)
ax.set_title(
    f'Decision Tree on SAE top-{TOP_K_PER_CLASS} features per class  |  '
    f'depth={MAX_DEPTH}  |  tree acc={(y_tree==y_true).mean()*100:.1f}%  |  '
    f'DeiT acc={(y_deit==y_true).mean()*100:.1f}%',
    fontsize=12,
)
plt.tight_layout()
plt.savefig('outputs/decision_tree_sae.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: outputs/decision_tree_sae.png')

# ── 7. Which features does the tree split on most? ────────────────────────
importances = clf.feature_importances_
top_cols    = np.argsort(importances)[::-1][:15]
print(f'\nTop-15 features by Gini importance (splits used by the tree):')
print(f"{'Rank':>4}  {'SAE feat':>8}  {'Importance':>10}  {'freq_img%':>10}  {'sel':>6}")
print('-' * 48)
for rank, col in enumerate(top_cols):
    f   = selected_feats[col]
    imp = importances[col]
    freq = class_freq_img[:, f].max() * 100
    sel  = selectivity[f]
    print(f'{rank+1:>4}  {f:>8}  {imp:>10.4f}  {freq:>9.1f}%  {sel:>6.3f}')